In [ ]:
"""
Objective: Use the PELT (Pruned Exact Linear Time) algorithm for endogenous detection 
of multiple structural breaks (regime shifts in mean and variance) in U.S. credit 
delinquency rates.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ruptures as rpt
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. DATA LOADING
# ==========================================
def load_pelt_data(csv_file: str) -> pd.DataFrame:
    """Loads the dataset using the maximum available time window to capture full break history."""
    print("[*] Loading clean dataset...")
    df = pd.read_csv(csv_file, index_col=0, parse_dates=True)
    df_model = df.loc['2000-01-01':'2026-04-01'].copy()
    return df_model

PORTFOLIOS = {
    'delinq_mortgages': 'Mortgages',
    'delinq_credit_cards': 'Credit Cards',
    'delinq_consumer': 'Consumer Credit'
}

# ==========================================
# 2. ALGORITHMIC DETECTION (PELT) & VISUALIZATION
# ==========================================
def run_pelt_and_plot(df: pd.DataFrame, portfolio_dict: dict, output_file: str):
    """
    Runs the PELT estimator with an optimal penalty to avoid overfitting (false crises)
    and plots the history highlighting the estimated structural shifts.
    """
    print("[*] Applying PELT algorithm for endogenous break detection...")
    
    fig, axes = plt.subplots(len(portfolio_dict), 1, figsize=(12, 12), sharex=True)
    
    for idx, (col, name) in enumerate(portfolio_dict.items()):
        series = df[col].dropna()
        signal = series.values.reshape(-1, 1)
        
        # PELT Algorithm (L2 cost function for mean/variance detection)
        algo = rpt.Pelt(model="l2", min_size=4).fit(signal)
        
        # BIC-style penalty to prevent overfitting (dependent on intrinsic variance and size)
        penalty = np.log(len(signal)) * signal.var()
        break_indices = algo.predict(pen=penalty)
        
        # The algorithm always returns the final index as a break; isolate only true shifts.
        break_dates = [series.index[i-1] for i in break_indices if i < len(series)]
        
        print(f" > {name}: {len(break_dates)} structural break(s) detected.")
        
        # Plot Time Series
        ax = axes[idx]
        ax.plot(series.index, series, color='#1f77b4', linewidth=2, label=f'Delinquency Rate ({name})')
        
        # Draw Break Lines and Date Labels
        for i, b_date in enumerate(break_dates):
            label_q = 'Structural Break (Regime Shift)' if i == 0 else ""
            ax.axvline(x=b_date, color='red', linestyle='--', linewidth=1.5, label=label_q)
            
            height_factor = 0.95 if i % 2 == 0 else 0.85
            y_height = ax.get_ylim()[1] * height_factor
            
            # Format: Q3 2008
            ax.text(b_date, y_height, f'  Q{b_date.quarter} {b_date.year}', 
                    color='red', fontweight='bold', fontsize=9, rotation=0, va='top', ha='left')
        
        ax.set_ylabel('Delinquency (%)', fontsize=10, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.legend(loc='upper left', frameon=False, fontsize=9)

    axes[-1].set_xlabel('Quarters', fontsize=11, fontweight='bold')
    plt.tight_layout()
    
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n[SUCCESS] Structural breaks chart exported as '{output_file}'.")

if __name__ == "__main__":
    df_model = load_pelt_data('clean_macro_fred_dataset.csv')
    run_pelt_and_plot(df_model, PORTFOLIOS, '05_credit_structural_breaks.png')